# Module 8 Assignment: Give Your Persona Agent Memory With Sessions

Lab 7 added a `SQLiteSession` to Study Buddy: first in memory, to fix two `Runner.run` calls that
did not remember each other, then file backed, so the conversation survives a kernel restart. The
Gradio `chat()` function got simpler in the process, no more rebuilding `history` by hand.

This assignment makes you do the same thing to your **own** Persona Agent, the one you rebuilt on
the OpenAI Agents SDK in the Module 7 assignment. Nothing about your agent's persona, notes, or
three tools needs to change here: you are giving it memory, not redesigning it.

**What you'll build:**
1. Your Module 7 assignment's Agent, Azure configuration, and three tools, carried over unchanged
2. A quick demonstration that two separate `Runner.run` calls do not remember each other
3. A file backed `SQLiteSession` that fixes it
4. A simplified Gradio `chat()` function that uses `session=session` instead of rebuilding `history`

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 7 made the same kind of change to Study Buddy, and do
the analogous thing to your own agent.

## 0. Install the packages (run once)

Same as Lab 7: no new package beyond what your Module 7 assignment already installed.
`SQLiteSession` ships with `openai-agents` itself.

In [1]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

Resolved 79 packages in 1.47s
Checked 79 packages in 10ms


## 1. Imports

**TODO:** Copy the imports from your Module 7 assignment, plus one new name: `SQLiteSession`,
imported from `agents` alongside `Agent` and `Runner`, the same way Lab 7, Step 1 added it.

In [50]:
# TODO: copy your imports from the Module 7 assignment, plus SQLiteSession from agents
from dotenv import load_dotenv
import os
import asyncio
import html
from typing import Literal, Optional

import requests
from openai import AsyncAzureOpenAI
from agents import ( Agent, Runner, function_tool, OpenAIChatCompletionsModel, set_tracing_disabled, SQLiteSession )
from pypdf import PdfReader
import gradio as gr
# #region agent log
import json, time, inspect
from typing import get_type_hints
from agents.function_schema import function_schema

## 2. Load your Azure OpenAI credentials from `.env`

Identical to the Module 7 assignment. Sessions do not change which provider you talk to.

In [51]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from your Module 7 assignment, unchanged: `set_tracing_disabled(True)`,
your `AsyncAzureOpenAI` client, and `OpenAIChatCompletionsModel`. A session is a feature of
`Runner.run`, not of the model or the client, so nothing here needs to change.

In [52]:
# TODO: set_tracing_disabled(True), then build azure_client and azure_model, exactly like your Module 7 assignment
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint= AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")


Azure model configured for the OpenAI Agents SDK.


## 4. Your persona, notes, and three tools

**TODO:** Bring over your persona, your notes, and all three of your `@function_tool` decorated
tools from the Module 7 assignment, unchanged. None of them need to change for this assignment: a
session only affects how conversation history is stored and replayed, not what a tool does.

In [53]:
# TODO: persona = "..." (a parenthesized multi-line string), from your Module 7 assignment
# TODO: if your persona uses notes, reader = PdfReader("notes/notes.pdf"); notes = ""; loop over
#       reader.pages and build up notes the same way Lab 6 did, if applicable
# TODO: persona = "..." and notes = "..." (or load_notes(...) if you used a PDF), from Module 6
persona = (
    "Lumi, A relaxed, chill and helpful study tutor. Is not too hard on you for mistakes and helps you build your understanding."

)

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text


print(persona)
print(notes[:300], "...")

Lumi, A relaxed, chill and helpful study tutor. Is not too hard on you for mistakes and helps you build your understanding.
Notes on Basketball 
• A “3 point shot” is a basket that is scored from beyond the 3-point line 
• A “free throw” is the compensation free shots from the free throw line after a 
shooting-foul is committed 
• A “Dunk” is when a player slams the basketball directly through the basket. 
• A “Center” i ...


In [54]:
# TODO: your first tool, decorated with @function_tool, from your Module 7 assignment
@function_tool
def track_score_tool(correct: str) -> str:
    """Log when a student has answered a question correctly.

    Args:
        correct: The question or concept the student answered correctly.
    """
    print(f"Tool called to log when a question is answered correctly: {correct}")
    with open("correct_answers.txt", "a", encoding="utf-8") as f:
        f.write(correct + "\n")
    return "Correct Answer logged"

print("Tool ready:", track_score_tool.name)

Tool ready: track_score_tool


In [55]:
# TODO: your second tool, decorated with @function_tool, from your Module 7 assignment
OPENTDB_CATEGORIES = {"sports": 21,}
@function_tool
def extra_sports_question_tool(category: Optional[Literal["sports"]] = None) -> str:
    """Fetch an extra sports trivia question from an external trivia database when the student asks for a bonus question or a break from the study notes.

    Args:
        category: Which trivia category to pull a bonus question from, if the student asked for one.
    """
    print(f"Tool called to fetch a extra basketball trivia question (category={category})")
    params = {"amount": 1, "type": "multiple"}
    if category in OPENTDB_CATEGORIES:
        params["category"] = OPENTDB_CATEGORIES[category]

    try:
        response = requests.get("https://opentdb.com/api.php", params=params, timeout=20)
        data = response.json()
    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"extra_sports_question_tool network error: {e}")
        return "The trivia API is temporarily unavailable - just skip the bonus question and continue the quiz."

    if data.get("response_code") != 0 or not data.get("results"):
        return "Could not fetch a bonus question right now - just skip it and continue the quiz."

    q = data["results"][0]
    question = html.unescape(q["question"])
    answer = html.unescape(q["correct_answer"])
    return f"Bonus question: {question} (correct answer: {answer})"

print("Tool ready:", extra_sports_question_tool.name)

Tool ready: extra_sports_question_tool


In [56]:
# TODO: your third tool, decorated with @function_tool, from your Module 7 assignment
@function_tool
def log_review_tool(topic: str) -> str:
    """Log a sports topic for later review.

    Args:
        topic: The topic of the question the student is answering.
    """
    print(f"Tool called to log a sports topic for review: {topic}")
    with open("sports_review_topics.txt", "a", encoding="utf-8") as f:
        f.write(topic + "\n")
    return "Review topic logged"

print("Tool ready:", log_review_tool.name)

Tool ready: log_review_tool


## 5. Build your Agent

**TODO:** Copy this from your Module 7 assignment, unchanged: `name`, `instructions`, your `tools`
list, and `model=azure_model`.

In [57]:
# TODO: instructions = f"""..."""  (your # Your role / # Context / # Rules structured template,
#       using {persona} and {notes}, from your Module 7 assignment)
# TODO: your_agent = Agent(name=..., instructions=instructions, tools=[...], model=azure_model)

instructions = f"""
# Your role

You are {persona}

You are to quiz the student using the content below

# Content

Here are the study notes you are quizzing the student on: 

{notes}

# Rules 

Go through the notes one at a time and ask a question to the student based on a bullet point in the notes, wait for the students answer before
asking another.

After the student gives his answer, tell him if he was right or wrong, and give a short explanation on either why he is right or why he was wrong.

Anytime the student is wrong, record it as a missed question using track_score_tool, and if they say "I dont know", then skip the question and move on to the next. 

ONLY ask questions about the notes (unless the student ask for a extra_sports_question_tool), then give them a extra sports question. 

After each answered question, log the topic of question that was answered using log_review_tool, then return "Review Topic Logged.


"""

Lumi = Agent(
    name="Lumi",
    instructions=instructions,
    tools=[track_score_tool, extra_sports_question_tool, log_review_tool],
    model=azure_model,
)


print("Agent built:", Lumi.name)


Agent built: Lumi


## 6. Confirm the problem: two calls that do not know each other

**TODO:** Run your agent twice in a row with `await Runner.run(your_agent, "...")`, the same way
Lab 7, Step 7 did with Study Buddy. Pick a first message that gives the agent some information (for
example, ask it a question whose answer it needs to remember), then a second message that only
makes sense if the agent remembers the first (for example, "what did I just ask you?" or "say that
again"). Confirm the second reply shows the agent has no idea what you mean.

In [58]:
# TODO: result = await Runner.run(your_agent, "...")
# TODO: print(result.final_output)

result = await Runner.run(Lumi, "Could you quiz me the 3pt shot question?")
print(result.final_output)

Sure — what is a “3 point shot” in basketball?


In [59]:
# TODO: result = await Runner.run(your_agent, "...a follow up that only makes sense with memory...")
# TODO: print(result.final_output)

result = await Runner.run(Lumi, "What question did I just ask you to quiz me on.")
print(result.final_output)

You haven’t answered a quiz question yet. We’re starting with the first note.

First question: What is a “3 point shot” in basketball?


## 7. Fix it with a file backed `SQLiteSession`

**TODO:** Create a `SQLiteSession`, the same way Lab 7, Step 9 did: give it a `session_id` string,
and a `db_path` so the conversation is written to a file and survives a kernel restart. Then repeat
the same two messages from Step 6, this time passing `session=your_session` to both calls, and
confirm the second reply now correctly remembers the first.

In [60]:
# TODO: your_session = SQLiteSession("your_session_id", "your_conversation_history.db")
your_session = SQLiteSession("demo_conversation")

result = await Runner.run(
    Lumi,
    "Could you quiz me the 3pt shot question?",
    session=your_session,
)
print(result.final_output)

Sure — what is a “3 point shot” in basketball?


In [61]:
# TODO: result = await Runner.run(your_agent, "...", session=your_session)
# TODO: print(result.final_output)
result = await Runner.run(
    Lumi,
    "What question did I just ask you to quiz me on.",
    session=your_session,
)
print(result.final_output)

You asked me to quiz you on the 3 point shot question.


In [62]:
# TODO: result = await Runner.run(your_agent, "...same follow up as Step 6...", session=your_session)
# TODO: print(result.final_output)
items = await your_session.get_items()
print(f"{len(items)} items stored in this session")
for item in items:
    print(item.get("role"), "-", str(item.get("content"))[:80])

4 items stored in this session
user - Could you quiz me the 3pt shot question?
assistant - [{'annotations': [], 'text': 'Sure — what is a “3 point shot” in basketball?', '
user - What question did I just ask you to quiz me on.
assistant - [{'annotations': [], 'text': 'You asked me to quiz you on the 3 point shot quest


## 8. Give it a real chat UI with Gradio

**TODO:** Copy this skeleton from Lab 7, Step 11 exactly, structure and all. Notice what is
different from your Module 7 assignment's version: `history` is no longer rebuilt inside `chat()`
at all, `session=your_session` already gives `Runner.run` everything it needs to remember the
conversation.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [49]:
async def chat(message, history):
    result = await Runner.run(Lumi, message, session=your_session)
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

KeyboardInterrupt: 

## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. In Step 6, what exactly did your agent get wrong on the second call, and why? Point to the
   specific piece of information it had no way of knowing.

   It literally could not remember that I had asked it anything, because it had no stored log of the conversation to look back on.

2. Compare your Gradio `chat()` function here to the one in your Module 7 assignment. What code did
   you get to delete, and what does `session=your_session` do in its place?

   we deleted the code that manually stored them memory, session=your_session does all of that within itself automatically, taking the content and the variables from the conversation and defining them. It makes it so we dont have to manually enter list comprehensions for the AI. 

3. Your `your_session` was created with a `db_path`. If you had left `db_path` out entirely, what
   would still work the same, and what would break the next time you restarted the kernel?

   It would work the same as long as the current state of the kernel is maintained, but if you restarted the kernel it would wipe the memory database because nothing gets written.

4. `SQLiteSession` also has a `pop_item()` method, demonstrated in Lab 7, Step 10, for undoing the
   last turn. Describe one realistic situation in your own agent's domain where you would want to
   call it.
   
   It would be useful for if you messed up the phrasing of the previous turn, or if you wanted to test a different prompt without the interference of the previous turns memory log.
   
## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Remove any `.db` file your session created; it may contain real conversation content.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm Step 6 shows the agent failing to remember, and Step 7 shows the same scenario working
  correctly once `session=your_session` is added.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module8Assignment_JaneDoe.ipynb`, then submit
  it.